In [ ]:
import os
import sys
import json
import glob

import pandas as pd
from openpyxl import load_workbook

# Make the shared cleaning package (backend/cleaning/) importable from this notebook
sys.path.insert(0, os.path.abspath(os.path.join("..", "backend")))

# NOTE: cleaning.master_json is deprecated (see that file) and not imported here.
from cleaning import io_utils, census_transform, proportions, recipes, pipeline

### Basic Utilities — now in `backend/cleaning/io_utils.py`

### Census Dataset Utilities — now in `backend/cleaning/census_transform.py`

### Dataset processing pipeline — now in `backend/cleaning/pipeline.py`

### load_dataset_config, process_census_dataset, load_and_process_all_datasets now live in backend/cleaning/pipeline.py

In [ ]:
path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/")
notes_wb_path = os.path.join(path_head, "Metrics/Notes/JPL_CCSVI_all_fields.xlsx")

In [ ]:
notes_wb = load_workbook(notes_wb_path)
sheet = notes_wb.active

In [ ]:
file_blocks = {}
current_file_name = None
cols_not_to_drop = ["Geography", "Geographic Area Name"] 

# Iterate through rows to find blocks for each csv file
for row in sheet.iter_rows(min_row=1, max_col=3, values_only=False):
    cell_value = row[0].value
    is_bold = row[0].font.bold if row[0].font else False

    # Detect file block by bold file name
    if is_bold and cell_value:
        original_name = cell_value.strip()
        current_file_name = io_utils.to_snake_case(original_name)
        file_blocks[current_file_name] = {
            'original_name': original_name,
            'cols_to_drop': [],
            'code_to_alias_column_mappings': {},
            'original_file_path': '',
            'centralized_file_dir': '' 
        }
        continue

    # Check for columns with a "Subfield to keep" value
    if current_file_name and any(col.value for col in row):

        col_name = row[0].value
        subfield_value = row[1].value
        # Track columns with no value in the subfields to keep column
        if col_name and not subfield_value:
            if col_name not in cols_not_to_drop:
                cleaned_col_name = col_name.strip().strip('\'"')
                cleaned_col_name = ' '.join(cleaned_col_name.split())
                file_blocks[current_file_name]['cols_to_drop'].append(cleaned_col_name)

In [ ]:
list(file_blocks.items())

In [ ]:
# exposures_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures")
central_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/JPL")
cleaned_path_head = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/cleaned-data")
json_dir_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/census-jsons")

In [ ]:
config_path = "../backend/cleaning/config/census_datasets_config.json"
base_path = "~/Desktop/Nextcloud/SCOVI Project/Metrics/"

In [ ]:
datasets = pipeline.load_and_process_all_datasets(config_path, base_path, file_blocks, central_path_head)

In [ ]:
# datasets.keys()

### Age of Structure

##### JPL Notes Cleaning

In [ ]:
age_of_structure_df = datasets['age_of_structure']
age_of_structure_df.head(2)

In [ ]:
cleaned_age_of_structure_df = recipes.recipe_age_of_structure(age_of_structure_df)
cleaned_age_of_structure_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_age_of_structure_df, cleaned_path_head, "age_of_structure.csv", True)

### Aggregate number of vehicles

In [ ]:
aggregate_vehicles_df = datasets['aggregate_vehicles']
aggregate_vehicles_df.head(2)

In [ ]:
census_transform.export_census_csv(aggregate_vehicles_df, cleaned_path_head, "aggregate_vehicles.csv", True)

### Health insurance

##### JPL Notes Cleaning

In [ ]:
health_insurance_df = datasets['health_insurance']
health_insurance_df.head(2)

In [ ]:
cleaned_health_insurance_df = recipes.recipe_health_insurance(health_insurance_df)
cleaned_health_insurance_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_health_insurance_df, cleaned_path_head, "health_insurance.csv", True)

### Households with a computer

In [ ]:
households_w_computer_df = datasets['households_w_computer']
households_w_computer_df.head(2)

In [ ]:
census_transform.export_census_csv(households_w_computer_df, cleaned_path_head, "households_w_computer.csv", True)

### Internet subscription

In [ ]:
internet_subscription_df = datasets['internet_subscription']
internet_subscription_df.head(2)

In [ ]:
census_transform.export_census_csv(internet_subscription_df, cleaned_path_head, "internet_subscription.csv", True)

### Limited English speaking

##### JPL Notes Cleaning

In [ ]:
limited_english_speaking_df = datasets['limited_english_speaking']
limited_english_speaking_df.head(2)

In [ ]:
cleaned_limited_english_speaking_df = recipes.recipe_limited_english_speaking(limited_english_speaking_df)
cleaned_limited_english_speaking_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_limited_english_speaking_df, cleaned_path_head, "limited_english_speaking.csv", True)

### Living Arrangements

##### JPL Notes Cleaning

In [ ]:
living_arrangements_df = datasets['living_arrangements']
living_arrangements_df.head(2)

In [ ]:
cleaned_living_arrangements_df = recipes.recipe_living_arrangements(living_arrangements_df)
cleaned_living_arrangements_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_living_arrangements_df, cleaned_path_head, "living_arrangements.csv", True)

### Income share of FPL

##### JPL Notes Cleaning

In [ ]:
income_share_of_fpl_df = datasets['income_share_of_fpl']
income_share_of_fpl_df.head(2)

In [ ]:
cleaned_fpl_df = recipes.recipe_income_share_of_fpl(income_share_of_fpl_df)
cleaned_fpl_df.head(3)

In [ ]:
census_transform.export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

### Persons under 5 & 65

##### JPL Notes Cleaning

In [ ]:
person_under_5_65_df = datasets['person_under_5_65']
person_under_5_65_df.head(2)

In [ ]:
person_under_5_65_outputs = recipes.recipe_person_under_5_65(person_under_5_65_df)
person_under_5_65_outputs['genders'].head(2)

In [ ]:
census_transform.export_census_csv(person_under_5_65_outputs['genders'], cleaned_path_head, "genders.csv", True)

In [ ]:
person_under_5_65_outputs['males'].head(2)

In [ ]:
census_transform.export_census_csv(person_under_5_65_outputs['males'], cleaned_path_head, "person_under_5_65_males.csv", True)

In [ ]:
person_under_5_65_outputs['females'].head(2)

In [ ]:
census_transform.export_census_csv(person_under_5_65_outputs['females'], cleaned_path_head, "person_under_5_65_females.csv", True)

### Population in group quarters

In [ ]:
population_group_quarters_df = datasets['population_group_quarters']
population_group_quarters_df.head(2)

In [ ]:
census_transform.export_census_csv(population_group_quarters_df, cleaned_path_head, "population_group_quarters.csv", True)

### Race origin

In [ ]:
race_origin_df = datasets['race_origin']
race_origin_df.head(2)

In [ ]:
census_transform.export_census_csv(race_origin_df, cleaned_path_head, "race_origin.csv", True)

### Tenure

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
census_transform.export_census_csv(tenure_df, cleaned_path_head, "tenure.csv", True)

### 2022 Census Hawaiian Homelands

In [ ]:
tenure_df = datasets['tenure']
tenure_df.head(2)

In [ ]:
# export_census_csv(tenure_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### JPL Notes Cleaning

In [ ]:
hawaiian_homelands_df = datasets['2022_census_hawaiian_homelands']
hawaiian_homelands_df.head(2)

In [ ]:
cleaned_hawaiian_homelands_df = recipes.recipe_hawaiian_homelands(hawaiian_homelands_df)
cleaned_hawaiian_homelands_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_hawaiian_homelands_df, cleaned_path_head, "2022_census_hawaiian_homelands.csv", True)

##### Add Hawaiian Homelands census poverty data to FPL poverty status dataset

In [ ]:
cleaned_fpl_df = recipes.merge_hawaiian_homelands_poverty(cleaned_fpl_df, hawaiian_homelands_df)
cleaned_fpl_df.head(2)

In [ ]:
census_transform.export_census_csv(cleaned_fpl_df, cleaned_path_head, "income_share_of_fpl.csv", True)

## Add Proportions to All Datasets ====================================

In [ ]:
# Block groups population from 2020 Census
block_groups_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/2020_Census_Block_Groups_Stripped.geojson")
hawaiian_homelands_geojson_path = os.path.expanduser("~/Desktop/Nextcloud/SCOVI Project/Metrics/All Exposures/census/block-groups/Census_Hawaiian_Homelands_hhl10_Stripped.geojson")

block_group_populations = proportions.load_block_group_populations(block_groups_geojson_path)
hawaiian_homelands_populations = proportions.load_hawaiian_homelands_populations(hawaiian_homelands_geojson_path)

total_population_block_groups = sum(block_group_populations.values())
total_population_hawaiian_homelands = sum(hawaiian_homelands_populations.values())
total_population = total_population_block_groups + total_population_hawaiian_homelands

print(f"Total population (block groups): {total_population_block_groups}")
print(f"Total population (Hawaiian homelands): {total_population_hawaiian_homelands}")
print(f"Total population (combined): {total_population}")

## ============================================================

In [ ]:
# import glob

# csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

# # Process each CSV file
# for csv_file in csv_files:
#     print(f"Processing {csv_file}...")
#     try:
#         census_csv_to_json(csv_file, json_dir_path)
#     except Exception as e:
#         print(f"Error processing {csv_file}: {e}")

In [ ]:
# Process all CSVs in the cleaned directory
csv_files = glob.glob(os.path.join(cleaned_path_head, "*.csv"))

for csv_file in csv_files:
    try:
        print(f"Adding proportions to {os.path.basename(csv_file)}...")
        df_with_props = proportions.add_percentages_to_csv(csv_file, total_population, block_group_populations, hawaiian_homelands_populations)
        
        # Save back to the same file
        df_with_props.to_csv(csv_file, index=False)
        print(f"  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")

## Leaflet JSON 

In [ ]:
# DEPRECATED: backend/cleaning/master_json.py (census_csvs_to_master_json, clean_column_name)
# built census_metrics_by_block_group.json for the old census_metrics table, which was
# migrated to the current normalized schema and dropped by backend/ingest/migrate_schema.py.
# Nothing reads that JSON anymore — see ADDING_DATASETS_GUIDE.md §2 for the current path
# (a one-off script that upserts cleaned data straight into datasets/metrics/metric_values).
# The function is commented out, not deleted, in master_json.py for reference.

# generate_dataset_params: replaced by per-map classification mode UI control.
# Pre-computing quantile thresholds and color schemes server-side is no longer
# needed — the frontend computes the color scale at render time based on the
# user's chosen classification mode.

In [ ]:
# DEPRECATED — see the note above. Nothing consumes this JSON anymore; kept commented for reference.
# json_output_path = os.path.join(json_dir_path,"census_metrics_by_block_group.json") 
# metrics = master_json.census_csvs_to_master_json(cleaned_path_head, json_output_path, block_group_populations, hawaiian_homelands_populations)

In [ ]:
# census_datasets_config.json generation removed — dataset metadata (labels,
# hawaiian_homelands flag, mappable columns) will be stored directly in the DB.
# Classification mode and color schemes are now per-map UI controls.
#
# json_output_path = os.path.join(json_dir_path, "census_datasets_config.json")
#
# dataset_params = generate_dataset_params(
#     cleaned_path_head,
#     json_output_path,
#     block_group_populations,
#     hawaiian_homelands_populations
# )